In [2]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader

In [3]:
def visualize_predictions(model, dataloader, label_encoder, output_dir, device, num_visualizations=5):
    """
    Visualizes per-residue predictions for a subset of the validation set,
    showing the *entire padded protein sequence*.

    Args:
        model (torch.nn.Module): The trained model.
        dataloader (DataLoader): DataLoader for the validation set.
        label_encoder (LabelEncoder): The LabelEncoder used during training.
        output_dir (str): Directory to save the visualization plots.
        device (torch.device): The device (CPU or MPS) to run inference on.
        num_visualizations (int): Number of proteins to visualize.
    """
    model.eval()
    os.makedirs(output_dir, exist_ok=True)

    # Reverse mapping from encoded ID to label string for plotting
    id_to_label = {i: label for i, label in enumerate(label_encoder.classes_)}

    visualized_count = 0

    with torch.no_grad():
        for i, (x, y_true) in enumerate(dataloader):
            if visualized_count >= num_visualizations:
                break

            for k, v in x.items():
                x[k] = v.to(device, non_blocking=True)
            y_true = y_true.to(device, non_blocking=True)

            outputs = model(x)
            # Permute to (batch_size, sequence_length, num_classes) if not already
            outputs = outputs.permute(0, 2, 1)

            # Get predicted class for each residue
            y_pred = torch.argmax(outputs, dim=1)

            # Process each protein in the batch
            for batch_idx in range(x["embedding"].shape[0]):
                if visualized_count >= num_visualizations:
                    break

                # The `full_display_length` is the length of the padded tensor
                full_display_length = x["embedding"].shape[1]  # Use the sequence length from the batch

                true_labels_tensor = y_true[batch_idx].cpu()
                pred_labels_tensor = y_pred[batch_idx].cpu()

                # No filtering based on original length; show the full padded sequence
                true_labels_to_plot = true_labels_tensor.numpy()
                pred_labels_to_plot = pred_labels_tensor.numpy()

                # Convert encoded IDs back to original labels for plotting
                true_labels_str = [id_to_label[int(id_val)] for id_val in true_labels_to_plot]
                pred_labels_str = [id_to_label[int(id_val)] for id_val in pred_labels_to_plot]

                # Create a simple visual representation
                fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 6), sharex=True)
                fig.suptitle(
                    f"Protein {i * dataloader.batch_size + batch_idx + 1} Residue-wise Prediction (Full Sequence)")

                residue_indices = np.arange(full_display_length)

                # Map labels to numerical values for plotting
                unique_labels = sorted(list(set(true_labels_str + pred_labels_str)))
                label_to_plot_value = {label: idx for idx, label in enumerate(unique_labels)}

                true_plot_values = [label_to_plot_value[label] for label in true_labels_str]
                pred_plot_values = [label_to_plot_value[label] for label in pred_labels_str]

                # Plot true labels
                ax1.imshow(np.array(true_plot_values).reshape(1, -1), cmap='tab20', aspect='auto',
                           extent=[0, full_display_length, 0, 1])
                ax1.set_yticks([])
                ax1.set_title("True CATH Domains")
                ax1.set_ylabel("True")
                ax1.set_xlim(0, full_display_length)  # Ensure x-axis covers the full length

                # Plot predicted labels
                ax2.imshow(np.array(pred_plot_values).reshape(1, -1), cmap='tab20', aspect='auto',
                           extent=[0, full_display_length, 0, 1])
                ax2.set_yticks([])
                ax2.set_title("Predicted CATH Domains")
                ax2.set_xlabel("Residue Index")
                ax2.set_ylabel("Predicted")
                ax2.set_xlim(0, full_display_length)  # Ensure x-axis covers the full length

                # Create a custom legend
                cmap_norm = len(unique_labels) - 1 if len(unique_labels) > 1 else 1
                handles = [plt.Rectangle((0, 0), 1, 1, color=plt.cm.tab20(label_to_plot_value[label] / cmap_norm)) for
                           label in unique_labels]
                ax2.legend(handles, unique_labels, loc='upper center', bbox_to_anchor=(0.5, -0.2),
                           fancybox=True, shadow=True, ncol=3)

                plt.tight_layout(rect=[0, 0.03, 1, 0.95])
                plt.savefig(os.path.join(output_dir, f"full_protein_prediction_{visualized_count + 1}.png"))
                plt.close(fig)

                visualized_count += 1
                print(f"Generated visualization for protein {visualized_count}")

In [4]:
from src.protenn2.model import CathPredEnn2
from src.protenn2.dataset import CathPredPerResidueDataset, create_protein_collate_fn
from src.protenn2.utils import get_train_val_test_paths, calculate_max_protein_length

# Define paths (adjust these to your specific project structure)
# IMPORTANT: Make sure these paths point to where your trained model,
# label encoder, and dataset CSVs/embeddings are located.
output_path = "../../output/protenn2/v2"
model_path = os.path.join(output_path, "best_model.pt")
label_encoder_path = os.path.join(output_path, "label_encoder.pkl")
input_data_folder = "../../datasets/v3"

visualization_output_dir = os.path.join(output_path, "visualization_results")

# Set device (MPS for Apple Silicon, otherwise CPU)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon GPU) for visualization.")
else:
    device = torch.device("cpu")
    print("MPS not available, falling back to CPU for visualization.")

# Load LabelEncoder
try:
    with open(label_encoder_path, "rb") as f:
        label_encoder = pickle.load(f)
    print(f"Loaded LabelEncoder with {len(label_encoder.classes_)} classes.")
except FileNotFoundError:
    print(f"Error: LabelEncoder file not found at {label_encoder_path}. Please check the path.")
    exit()  # Exit if the label encoder isn't found

# Get data paths
train_path, val_path, test_path = get_train_val_test_paths(input_data_folder)

# Create validation dataset and dataloader
# Pass the correct embedding_dir to the dataset constructor
val_dataset = CathPredPerResidueDataset(val_path, label_encoder,
                                        embedding_dir="../../data/embeddings/protein_embeddings", fit=False)
max_protein_length = calculate_max_protein_length(input_data_folder)

if max_protein_length == 0:
    print(
        "Error: Max protein length is 0. This usually means no valid protein embeddings were found or data paths are incorrect.")
    print(
        "Please ensure 'input_data_folder' and 'embedding_base_dir' are correctly set and contain the necessary files.")
    exit()

collate_fn = create_protein_collate_fn(max_protein_length, val_dataset.no_domain_encoded_id)
# Use batch_size=1 for easier visualization of individual proteins
val_dataloader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

# Initiate and load the model
num_classes = len(label_encoder.classes_)
model = CathPredEnn2(num_classes=num_classes)
try:
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    print("Model loaded successfully.")
except FileNotFoundError:
    print(f"Error: Model file not found at {model_path}. Please check the path.")
    exit()
except Exception as e:
    print(f"Error loading model: {e}. Ensure CathPredEnn2 model definition matches the saved state_dict.")
    exit()

# Perform and visualize predictions
print("\nStarting visualization...")
visualize_predictions(model, val_dataloader, label_encoder, visualization_output_dir, device, num_visualizations=100)
print(f"\nVisualizations saved to {visualization_output_dir}")

Using MPS (Apple Silicon GPU) for visualization.
Loaded LabelEncoder with 803 classes.
Max protein length: 599
Model loaded successfully.

Starting visualization...
Generated visualization for protein 1
Generated visualization for protein 2
Generated visualization for protein 3
Generated visualization for protein 4
Generated visualization for protein 5
Generated visualization for protein 6
Generated visualization for protein 7
Generated visualization for protein 8
Generated visualization for protein 9
Generated visualization for protein 10
Generated visualization for protein 11
Generated visualization for protein 12
Generated visualization for protein 13
Generated visualization for protein 14
Generated visualization for protein 15
Generated visualization for protein 16
Generated visualization for protein 17
Generated visualization for protein 18
Generated visualization for protein 19
Generated visualization for protein 20
Generated visualization for protein 21
Generated visualization f